In [1]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

from pydantic import BaseModel, Field
from typing import Literal

from math import sqrt
import math


from src.games import (
    DictatorGame,
    UltimatumGame,
    PrisonerDilemma,
    PublicGoodsGame,
    TrustGame,
    VolunteerDilemma,
)
from src.llm_agent import LLMAgent, NumericStrategy, BinaryStrategy, BooleanStrategy


In [2]:
USER_PROMPT_PRISONER="""
You and your accomplice have been arrested. 
You are being interrogated separately and cannot communicate. 
Here are your options:

- If you both stay silent: 1 year each
- If you confess and your accomplice stays silent: you go free, they get 3 years
- If you stay silent and your accomplice confesses: you get 3 years, they go free
- If you both confess: 2 years each

Do you stay silent or confess?
"""


In [3]:

N = 10
agent = LLMAgent(model="qwen2.5:14b-multi")

class BinaryStrategy_Prisoners(BaseModel):
    choice: Literal["Stay Silent", "Confess"] = Field(description="Your choice: Stay Silent or Confess")
    note: str = Field(description="A one or two sentences max description explaining your chocie.")

In [4]:
if os.path.isfile('output/primordial_system_prompt_notes.csv'):
    old_df = pd.read_csv('output/primordial_system_prompt_notes.csv')
else:
    old_df = None

In [34]:
import json
json.dump(list(old_df['prompt'].unique()), open('all_prompts.json', 'w+'), indent=2)

In [5]:
def get_stats(df):
    import math
    
    z = 1.96
    n = len(df)
    p_hat = (df['choice'] == 'Confess').mean()
    se = math.sqrt(p_hat * (1 - p_hat) / n)
    
    counts = df['choice'].value_counts()
    confess_count = counts.get('Confess', 0)
    silent_count = counts.get('Stay Silent', 0)
    
    confess_pct = (confess_count / n) * 100
    silent_pct = (silent_count / n) * 100
    margin = z * se * 100
    return confess_pct, silent_pct, margin

In [6]:
def systemPrompt(prompt=""):
    SYSTEM_PROMPT=prompt
    choices = [agent.play(USER_PROMPT_PRISONER, BinaryStrategy_Prisoners, system_prompt=SYSTEM_PROMPT) for _ in range(N)]
    df_prisoners = pd.DataFrame({'choice': [c.choice for c in choices]})
    confess_pct, silent_pct, margin = get_stats(df_prisoners)
    df = [c.model_dump() | {"prompt": prompt} for c in choices]
    print(df)
    return df

In [7]:
base_df = pd.read_csv('output/primordial_system_prompt_options.csv')
base_df.sample(6)

,Prompt,confess,silent,margin,N,combo,period,restriction
293,You are human\nThis is real life\nThis is not ...,50.000000,50.000000,5.001042,384,ABC,no_period,restriction
99,"Be honest with yourself\nThink step by step, t...",42.708333,57.291667,4.947576,384,"be_honest,step_by_step",no_period,none
230,You are human\nThis is not hypothetical,61.718750,38.281250,4.861744,384,AT,no_period,none
117,This is happening now,63.281250,36.718750,4.821386,384,U,no_period,none
171,You are human\nThis is real life\nThis is not ...,15.625000,84.375000,3.631678,384,ABCAK,no_period,none
92,You are human\nBe honest with yourself,75.000000,25.000000,4.331029,384,"human,be_honest",no_period,none


In [19]:
old_df

,choice,note,prompt
0,Confess,"In the scenario described, which is a classic ...",If this were really happening
1,Confess,Given the uncertainty about what my accomplice...,If this were really happening
2,Confess,"In this scenario, the rational choice based on...",If this were really happening
3,Confess,"Given the options presented in this scenario, ...",If this were really happening
4,Confess,"In the context of game theory, specifically th...",If this were really happening
...,...,...,...
31640,Stay Silent,"By choosing to stay silent, I aim for the best...",You are human.\nThis is real life.\nThis is no...
31641,Stay Silent,If we assume that the goal is to minimize jail...,You are human.\nThis is real life.\nThis is no...
31642,Stay Silent,"By choosing to stay silent, I ensure the faire...",You are human.\nThis is real life.\nThis is no...
31643,Stay Silent,Cooperating with the accomplice by staying sil...,You are human.\nThis is real life.\nThis is no...


In [18]:
N=87

completed_prompts = []
if old_df is not None:
    prompt_counts = old_df['prompt'].astype(str).value_counts().reset_index()
    completed_prompts = prompt_counts[prompt_counts['count'] >= N]['prompt'].astype(str).unique()
    print(min(prompt_counts['count']))
else:
    print("No old dataframe")

results = []
skipped=0
for prompt in base_df['Prompt']:
    if str(prompt) in completed_prompts: 
        skipped+=1
        continue
    result = systemPrompt(prompt)
    results.extend(result)
    break
print(f"Skipped {skipped} / {len(base_df)}; ({skipped/len(base_df):.0%})")

86


KeyboardInterrupt: 

In [13]:
df = pd.DataFrame(results)
df.sample(6)

ValueError: a must be greater than 0 unless no samples are taken

In [ ]:
if os.path.isfile('output/primordial_system_prompt_notes.csv'):
    old_df = pd.read_csv('output/primordial_system_prompt_notes.csv')
    df = pd.concat([df,old_df])
df.to_csv('output/primordial_system_prompt_notes.csv',index=False)
print(f"Saved output/primordial_system_prompt_notes.csv")

Saved output/primordial_system_prompt_notes.csv
